In [2]:
import os

import cv2
import numpy as np
from tqdm import tqdm


# ==========================
# 設定
# ==========================

green_image_dir = "preprocessed_green_images"
original_image_dir = "healthy_images"

output_dir = "vessel_output"
mask_output_dir = os.path.join(output_dir, "masks")
skeleton_output_dir = os.path.join(output_dir, "skeletons")
overlay_output_dir = os.path.join(output_dir, "overlays")

# --- チューニング用パラメータ ---

# Black-hatのカーネルサイズ（小さい値を足すと細い血管を拾いやすくなる）
BLACKHAT_KERNEL_SIZES = [5, 7, 9, 13, 17, 21, 25]

# Frangiのsigma（小さい値を足すと細い血管を拾いやすくなる）
FRANGI_SIGMAS = [0.5, 0.75, 1, 1.5, 2, 3, 4]

# Hysteresis閾値（薄い血管を残すため元コードより緩和）
HYSTERESIS_LOW_PERCENTILE = 72      # 元: 80
HYSTERESIS_HIGH_PERCENTILE = 90     # 元: 92
HYSTERESIS_SEED_PERCENTILE = 55     # 元: 65
HYSTERESIS_MIN_COMPONENT_AREA = 6   # 元: 8

# 枝刈り（薄い・短い血管を消しすぎないよう緩和）
PRUNE_MIN_BRANCH_LENGTH = 6         # 元: 8
PRUNE_MIN_MEAN_SCORE = 18           # 元: 25

# ギャップ橋渡し（新規：鎖状の途切れを繋ぐ）
BRIDGE_MAX_GAP = 14          # 端点同士を繋ぐ最大距離(px)
BRIDGE_MAX_ANGLE_DEG = 35    # 端点の向きと接続線の角度差の許容値
BRIDGE_LOOKBACK = 6          # 端点の向きを推定するために遡るスケルトン画素数

# FOVマスク（境界付近の誤検出除去）
FOV_ERODE_PX = 12

# 細長さフィルタ（ブロブ状ノイズ除去）
ELONGATION_MIN_AREA_TO_CHECK = 40   # これより小さい成分だけ細長さをチェック
ELONGATION_MIN_ASPECT_RATIO = 1.6   # これ未満（丸い）は除去


# ==========================
# ファイル名変換
# ==========================

def get_original_file_name(preprocessed_file):
    if preprocessed_file.endswith("_green_preprocessed.png"):
        image_id = preprocessed_file.replace("_green_preprocessed.png", "")
    else:
        image_id = os.path.splitext(preprocessed_file)[0]
    return image_id + ".png"


def get_image_id(preprocessed_file):
    if preprocessed_file.endswith("_green_preprocessed.png"):
        return preprocessed_file.replace("_green_preprocessed.png", "")
    return os.path.splitext(preprocessed_file)[0]


# ==========================
# Percentile正規化
# ==========================

def percentile_normalize(image, lower_percentile=1, upper_percentile=99):
    output = np.zeros_like(image, dtype=np.uint8)
    values = image[image > 0]

    if values.size == 0:
        return output

    low = np.percentile(values, lower_percentile)
    high = np.percentile(values, upper_percentile)

    if high <= low:
        return output

    clipped = np.clip(image, low, high)
    normalized = (clipped - low) / (high - low) * 255
    normalized = normalized.astype(np.uint8)

    output[image > 0] = normalized[image > 0]
    return output


# ==========================
# FOVマスク（改善2: 境界の誤検出除去）
# ==========================

def compute_fov_mask(green, erode_px=FOV_ERODE_PX):
    """
    有効な眼底領域（FOV）のマスクを作成し、境界付近を収縮させる。

    眼底画像は円形の有効領域の外側が真っ黒（もしくは無効値）になっており、
    その境界のコントラスト段差をBlack-hat/Frangiが「血管」として誤検出
    しやすい。境界から内側に一定pxだけ収縮させることでこれを防ぐ。
    """

    _, raw_mask = cv2.threshold(green, 10, 255, cv2.THRESH_BINARY)

    # 小さい穴やノイズを除去してから最大の連結成分（＝眼底本体）のみ残す
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        raw_mask, connectivity=8
    )

    if num_labels <= 1:
        return raw_mask

    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    fov_mask = np.zeros_like(raw_mask)
    fov_mask[labels == largest_label] = 255

    if erode_px > 0:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (erode_px * 2 + 1, erode_px * 2 + 1)
        )
        fov_mask = cv2.erode(fov_mask, kernel)

    return fov_mask


# ==========================
# コントラスト強調（改善3: 薄い血管を強調）
# ==========================

def enhance_contrast(green):
    """
    CLAHEで局所的なコントラストを強調し、薄い血管も検出しやすくする。
    """

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    return clahe.apply(green)


# ==========================
# Frangi Filter（改善3: 細いsigmaを追加）
# ==========================

def frangi_vesselness(green, sigmas=FRANGI_SIGMAS):
    try:
        from skimage.filters import frangi
    except Exception:
        print("skimage が使えないため Frangi filter をスキップします。")
        return np.zeros_like(green, dtype=np.uint8)

    image_float = green.astype(np.float32) / 255.0

    vesselness = frangi(image_float, sigmas=sigmas, black_ridges=True)
    vesselness = np.nan_to_num(vesselness)

    if vesselness.max() <= vesselness.min():
        return np.zeros_like(green, dtype=np.uint8)

    vesselness = (
        (vesselness - vesselness.min())
        / (vesselness.max() - vesselness.min())
        * 255
    )

    return vesselness.astype(np.uint8)


# ==========================
# Hysteresis Threshold
# ==========================

def hysteresis_threshold_with_seed(
    score_image,
    seed_image,
    low_percentile=HYSTERESIS_LOW_PERCENTILE,
    high_percentile=HYSTERESIS_HIGH_PERCENTILE,
    seed_percentile=HYSTERESIS_SEED_PERCENTILE,
    min_component_area=HYSTERESIS_MIN_COMPONENT_AREA,
):
    score_values = score_image[score_image > 0]
    seed_values = seed_image[seed_image > 0]

    if score_values.size == 0:
        return np.zeros_like(score_image, dtype=np.uint8)

    if seed_values.size == 0:
        seed_values = score_values

    low_th = np.percentile(score_values, low_percentile)
    high_th = np.percentile(score_values, high_percentile)
    seed_th = np.percentile(seed_values, seed_percentile)

    low_mask = np.zeros_like(score_image, dtype=np.uint8)
    strong_mask = np.zeros_like(score_image, dtype=np.uint8)

    low_mask[score_image >= low_th] = 255
    strong_mask[(score_image >= high_th) & (seed_image >= seed_th)] = 255

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        low_mask, connectivity=8
    )

    output = np.zeros_like(score_image, dtype=np.uint8)

    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        if area < min_component_area:
            continue

        component = labels == label
        if np.any(strong_mask[component] > 0):
            output[component] = 255

    return output


# ==========================
# 線状カーネルで接続（改善1: カーネル拡大・角度追加）
# ==========================

def connect_vessel_segments(binary_image):
    """
    途切れた血管候補を、複数方向の線状カーネルで接続する。
    元コードは4方向・5pxだったが、鎖状の途切れが残っていたため
    7px・6方向に拡張して接続力を強化する。
    """

    kernels = []
    size = 7

    for angle_deg in [0, 30, 60, 90, 120, 150]:
        kernel = np.zeros((size, size), np.uint8)
        center = size // 2
        theta = np.deg2rad(angle_deg)
        dx = np.cos(theta)
        dy = np.sin(theta)

        for t in range(-center, center + 1):
            x = int(round(center + t * dx))
            y = int(round(center + t * dy))
            if 0 <= x < size and 0 <= y < size:
                kernel[y, x] = 1

        kernels.append(kernel)

    connected = binary_image.copy()

    for kernel in kernels:
        closed = cv2.morphologyEx(binary_image, cv2.MORPH_CLOSE, kernel)
        connected = cv2.bitwise_or(connected, closed)

    return connected


# ==========================
# 小領域除去
# ==========================

def remove_tiny_components(binary_image, min_area=6):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_image, connectivity=8
    )

    output = np.zeros_like(binary_image)

    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        if area >= min_area:
            output[labels == label] = 255

    return output


# ==========================
# 細長さフィルタ（改善2: 丸いノイズの除去）
# ==========================

def filter_by_elongation(
    binary_image,
    min_area_to_check=ELONGATION_MIN_AREA_TO_CHECK,
    min_aspect_ratio=ELONGATION_MIN_ASPECT_RATIO,
):
    """
    面積が小さく、かつ丸に近い（＝血管らしくない）成分を除去する。
    大きい成分（＝血管網である可能性が高い）は形状に関わらず残す。
    """

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_image, connectivity=8
    )

    output = np.zeros_like(binary_image)

    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]

        if area >= min_area_to_check:
            output[labels == label] = 255
            continue

        component_mask = (labels == label).astype(np.uint8) * 255
        contours, _ = cv2.findContours(
            component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        if len(contours) == 0:
            continue

        contour = max(contours, key=cv2.contourArea)

        if len(contour) < 5:
            # minAreaRectに十分な点がない場合は形状判定できないため残す
            output[labels == label] = 255
            continue

        (_, _), (w, h), _ = cv2.minAreaRect(contour)
        short_side = min(w, h)
        long_side = max(w, h)

        aspect_ratio = long_side / short_side if short_side > 0 else 0

        if aspect_ratio >= min_aspect_ratio:
            output[labels == label] = 255

    return output


# ==========================
# 細線化
# ==========================

def skeletonize_binary(binary_image):
    binary = binary_image > 0

    try:
        from skimage.morphology import skeletonize

        skeleton = skeletonize(binary)
        return (skeleton.astype(np.uint8)) * 255
    except Exception:
        pass

    img = binary.astype(np.uint8) * 255
    skeleton = np.zeros_like(img)
    element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))

    while True:
        opened = cv2.morphologyEx(img, cv2.MORPH_OPEN, element)
        temp = cv2.subtract(img, opened)
        eroded = cv2.erode(img, element)
        skeleton = cv2.bitwise_or(skeleton, temp)
        img = eroded.copy()

        if cv2.countNonZero(img) == 0:
            break

    return skeleton


# ==========================
# スケルトン共通処理
# ==========================

def get_skeleton_neighbors(y, x, skeleton_bool):
    h, w = skeleton_bool.shape
    neighbors = []

    for dy in [-1, 0, 1]:
        for dx in [-1, 0, 1]:
            if dy == 0 and dx == 0:
                continue
            ny, nx = y + dy, x + dx
            if 0 <= ny < h and 0 <= nx < w and skeleton_bool[ny, nx]:
                neighbors.append((ny, nx))

    return neighbors


def prune_short_skeleton_branches(
    skeleton_image,
    vessel_score,
    min_branch_length=PRUNE_MIN_BRANCH_LENGTH,
    min_mean_score=PRUNE_MIN_MEAN_SCORE,
):
    skeleton_bool = skeleton_image > 0

    if np.sum(skeleton_bool) == 0:
        return skeleton_image.copy()

    degree = np.zeros_like(skeleton_image, dtype=np.uint8)
    ys, xs = np.where(skeleton_bool)

    for y, x in zip(ys, xs):
        degree[y, x] = len(get_skeleton_neighbors(y, x, skeleton_bool))

    endpoints = list(zip(*np.where(skeleton_bool & (degree == 1))))

    remove_mask = np.zeros_like(skeleton_image, dtype=bool)
    visited_edges = set()

    for endpoint in endpoints:
        path = []
        current = endpoint
        previous = None

        while True:
            cy, cx = current
            path.append(current)

            neighbors = get_skeleton_neighbors(cy, cx, skeleton_bool)

            if previous is not None:
                neighbors = [nb for nb in neighbors if nb != previous]

            if len(neighbors) == 0:
                break

            if previous is not None and degree[cy, cx] != 2:
                break

            next_pixel = neighbors[0]
            edge_key = tuple(sorted([current, next_pixel]))

            if edge_key in visited_edges:
                break

            visited_edges.add(edge_key)
            previous = current
            current = next_pixel

            if len(path) > 10000:
                break

        if len(path) == 0:
            continue

        path_y = np.array([p[0] for p in path])
        path_x = np.array([p[1] for p in path])

        branch_length = len(path)
        mean_score = float(np.mean(vessel_score[path_y, path_x]))

        if branch_length < min_branch_length and mean_score < min_mean_score:
            remove_mask[path_y, path_x] = True

    pruned = skeleton_image.copy()
    pruned[remove_mask] = 0
    return pruned


def remove_tiny_skeleton_components(skeleton_image, min_pixels=5):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        skeleton_image, connectivity=8
    )

    output = np.zeros_like(skeleton_image)

    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        if area >= min_pixels:
            output[labels == label] = 255

    return output


# ==========================
# ギャップ橋渡し（新規: 改善1「鎖状の途切れ」対策）
# ==========================

def bridge_skeleton_gaps(
    skeleton_image,
    max_gap=BRIDGE_MAX_GAP,
    max_angle_deg=BRIDGE_MAX_ANGLE_DEG,
    lookback=BRIDGE_LOOKBACK,
):
    """
    細線化後、近接していて向きがほぼ同じ端点同士を線分で接続する。

    Hysteresis+接続処理でも埋まらなかった小さなギャップ（鎖状の途切れ）を
    ここで最終的に橋渡しする。無関係な血管同士を誤って繋がないよう、
    「距離が近い」かつ「端点付近の向きがほぼ一直線」の場合のみ接続する。
    """

    skeleton_bool = skeleton_image > 0
    ys, xs = np.where(skeleton_bool)

    if len(ys) == 0:
        return skeleton_image.copy()

    degree = np.zeros_like(skeleton_image, dtype=np.uint8)
    for y, x in zip(ys, xs):
        degree[y, x] = len(get_skeleton_neighbors(y, x, skeleton_bool))

    endpoints = list(zip(*np.where(skeleton_bool & (degree == 1))))

    def estimate_direction(endpoint):
        """端点から内側に向かって数画素遡り、血管の向きベクトルを推定する。"""
        current = endpoint
        previous = None
        trail = [current]

        for _ in range(lookback):
            cy, cx = current
            neighbors = get_skeleton_neighbors(cy, cx, skeleton_bool)
            if previous is not None:
                neighbors = [nb for nb in neighbors if nb != previous]

            if len(neighbors) == 0:
                break

            previous = current
            current = neighbors[0]
            trail.append(current)

        if len(trail) < 2:
            return None

        start = np.array(trail[-1], dtype=np.float32)
        end = np.array(trail[0], dtype=np.float32)
        vec = end - start

        norm = np.linalg.norm(vec)
        if norm == 0:
            return None

        return vec / norm

    bridged = skeleton_image.copy()
    n = len(endpoints)

    for i in range(n):
        p1 = np.array(endpoints[i], dtype=np.float32)
        dir1 = estimate_direction(endpoints[i])

        if dir1 is None:
            continue

        best_j = None
        best_dist = max_gap + 1

        for j in range(n):
            if i == j:
                continue

            p2 = np.array(endpoints[j], dtype=np.float32)
            dist = np.linalg.norm(p2 - p1)

            if dist <= 0 or dist > max_gap:
                continue

            connect_vec = (p2 - p1) / dist
            angle = np.degrees(
                np.arccos(np.clip(np.dot(dir1, connect_vec), -1.0, 1.0))
            )

            if angle > max_angle_deg:
                continue

            if dist < best_dist:
                best_dist = dist
                best_j = j

        if best_j is not None:
            p2 = endpoints[best_j]
            cv2.line(
                bridged,
                (int(p1[1]), int(p1[0])),
                (int(p2[1]), int(p2[0])),
                255,
                1,
            )

    return bridged


# ==========================
# 血管抽出メイン処理
# ==========================

def extract_vessels(green):
    """
    前処理済み緑チャンネル画像から血管を抽出する。

    流れ:
    green
        -> CLAHEでコントラスト強調
        -> FOVマスク作成（境界除外）
        -> Black-hat
        -> Frangi Filter
        -> 重み付き統合
        -> Hysteresis Threshold（FOVマスク適用）
        -> 線状カーネルで接続
        -> 小領域除去
        -> 細長さフィルタ（丸いノイズ除去）
        -> 細線化
        -> 短いスケルトン枝の除去
        -> ギャップ橋渡し（鎖状の途切れを接続）
    """

    fov_mask = compute_fov_mask(green)

    green_enhanced = enhance_contrast(green)
    green_smooth = cv2.GaussianBlur(green_enhanced, (3, 3), 0)

    # --- Black-hat ---
    blackhat_total = np.zeros_like(green_smooth)

    for kernel_size in BLACKHAT_KERNEL_SIZES:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (kernel_size, kernel_size)
        )
        blackhat = cv2.morphologyEx(green_smooth, cv2.MORPH_BLACKHAT, kernel)
        blackhat_total = cv2.max(blackhat_total, blackhat)

    blackhat_normalized = percentile_normalize(blackhat_total)

    # --- Frangi ---
    frangi_img = frangi_vesselness(green_smooth)
    frangi_normalized = percentile_normalize(frangi_img)

    # --- 重み付き統合 ---
    vessel_score = cv2.addWeighted(
        blackhat_normalized, 0.55, frangi_normalized, 0.45, 0
    )
    vessel_score = percentile_normalize(vessel_score)

    # FOV境界の外側を除外（改善2）
    vessel_score = cv2.bitwise_and(vessel_score, vessel_score, mask=fov_mask)
    frangi_normalized_masked = cv2.bitwise_and(
        frangi_normalized, frangi_normalized, mask=fov_mask
    )

    # --- Hysteresis ---
    vessel_raw = hysteresis_threshold_with_seed(
        score_image=vessel_score,
        seed_image=frangi_normalized_masked,
    )

    # --- 接続（改善1） ---
    vessel_connected = connect_vessel_segments(vessel_raw)
    vessel_connected = cv2.bitwise_and(
        vessel_connected, vessel_connected, mask=fov_mask
    )

    # --- 小領域除去 ---
    vessel_small_removed = remove_tiny_components(vessel_connected, min_area=6)

    # --- 細長さフィルタ（改善2） ---
    vessel_filtered = filter_by_elongation(vessel_small_removed)

    # --- 細線化 ---
    vessel_skeleton = skeletonize_binary(vessel_filtered)

    # --- 枝刈り（改善3: 緩和済み） ---
    vessel_skeleton = prune_short_skeleton_branches(
        skeleton_image=vessel_skeleton,
        vessel_score=vessel_score,
    )
    vessel_skeleton = remove_tiny_skeleton_components(vessel_skeleton, min_pixels=5)

    # --- ギャップ橋渡し（改善1） ---
    vessel_skeleton = bridge_skeleton_gaps(vessel_skeleton)

    return vessel_filtered, vessel_skeleton


# ==========================
# オーバーレイ画像作成
# ==========================

def create_overlay(original_img, vessel_skeleton):
    overlay = original_img.copy()

    vessel_colored = np.zeros_like(original_img)
    vessel_colored[:, :, 2] = vessel_skeleton

    overlay = cv2.addWeighted(overlay, 0.75, vessel_colored, 0.45, 0)
    return overlay


# ==========================
# メイン処理
# ==========================

def main():
    os.makedirs(mask_output_dir, exist_ok=True)
    os.makedirs(skeleton_output_dir, exist_ok=True)
    os.makedirs(overlay_output_dir, exist_ok=True)

    files = sorted(
        [f for f in os.listdir(green_image_dir) if f.lower().endswith(".png")]
    )

    if len(files) == 0:
        raise FileNotFoundError(f"{green_image_dir} にpng画像がありません。")

    for file in tqdm(files):
        green_path = os.path.join(green_image_dir, file)
        green = cv2.imread(green_path, cv2.IMREAD_GRAYSCALE)

        if green is None:
            print(f"画像を読み込めませんでした: {green_path}")
            continue

        original_file = get_original_file_name(file)
        original_path = os.path.join(original_image_dir, original_file)
        original_img = cv2.imread(original_path)

        if original_img is None:
            original_img = cv2.cvtColor(green, cv2.COLOR_GRAY2BGR)

        vessel_mask, vessel_skeleton = extract_vessels(green)

        image_id = get_image_id(file)

        overlay = create_overlay(original_img, vessel_skeleton)

        cv2.imwrite(
            os.path.join(mask_output_dir, f"{image_id}_vessel_mask.png"),
            vessel_mask,
        )
        cv2.imwrite(
            os.path.join(skeleton_output_dir, f"{image_id}_vessel_skeleton.png"),
            vessel_skeleton,
        )
        cv2.imwrite(
            os.path.join(overlay_output_dir, f"{image_id}_overlay.png"),
            overlay,
        )

    print("血管抽出が完了しました。")
    print(f"マスク画像: {mask_output_dir}")
    print(f"スケルトン画像: {skeleton_output_dir}")
    print(f"オーバーレイ画像: {overlay_output_dir}")


if __name__ == "__main__":
    main()


100%|██████████| 1434/1434 [20:13:21<00:00, 50.77s/it]    

血管抽出が完了しました。
マスク画像: vessel_output/masks
スケルトン画像: vessel_output/skeletons
オーバーレイ画像: vessel_output/overlays
